## 1. Cấu hình đường dẫn code & cài đặt phụ thuộc

In [ ]:
import os
import sys
import shutil
from pathlib import Path

is_kaggle = os.path.exists("/kaggle/working")
print(f"[+] Môi trường: {'KAGGLE' if is_kaggle else 'LOCAL'}")

if is_kaggle:
    # Trỏ sys.path tới thư mục dataset chứa code (nơi có config.py)
    # TODO: Đổi tên dataset code của bạn trên Kaggle nếu cần
    code_dir_candidates = [
        Path("/kaggle/input/mid-fusion-code"),
    ]
    code_dir = next((c for c in code_dir_candidates if (c / "config.py").exists()), None)
    if code_dir:
        sys.path.insert(0, str(code_dir))
        print(f"[+] Đã trỏ sys.path tới thư mục code: {code_dir}")
    else:
        print("[!] CẢNH BÁO: Không tìm thấy thư mục chứa code (config.py) trong /kaggle/input.")
else:
    # Local: notebook nằm cùng thư mục với config.py / runner.py
    sys.path.insert(0, os.getcwd())
    print(f"[+] sys.path local: {os.getcwd()}")

[+] Môi trường: KAGGLE
[+] Đã trỏ sys.path tới thư mục code: /kaggle/input/datasets/khiemdangle/isic-02-07


In [2]:
# (Tuỳ chọn) cài Albumentations / timm nếu môi trường thiếu.
# Trên Kaggle thường đã có sẵn — bỏ qua cell này nếu không cần.
# !pip install -q albumentations timm

## 2. Import pipeline & hàm dò dữ liệu ISIC

In [3]:
from config import MidFusionLiteConfig, resolve_data_paths
from runner import run_mid_fusion_cv


def resolve_isic_data_paths(config):
    """Tự động tìm bộ dữ liệu ISIC 2024.

    - Kaggle: quét /kaggle/input để tìm train-metadata.csv.
    - Local : dùng resolve_data_paths của config.py.
    """
    if not is_kaggle:
        return resolve_data_paths(config)

    print("\n[+] Đang dò bộ dữ liệu ISIC 2024 trong /kaggle/input ...")
    dataset_root = None
    for root, _dirs, files in os.walk("/kaggle/input/competitions/isic-2024-challenge"):
        if "train-metadata.csv" in files:
            dataset_root = Path(root)
            break

    if dataset_root is None:
        raise FileNotFoundError(
            "Không tìm thấy train-metadata.csv trong /kaggle/input. "
            "Hãy 'Add Data' bộ dataset ISIC 2024 vào notebook!"
        )

    print(f"    -> Tìm thấy dữ liệu tại: {dataset_root}")
    return config.with_updates(
        train_metadata_path=str(dataset_root / "train-metadata.csv"),
        test_metadata_path=str(dataset_root / "test-metadata.csv"),
        sample_submission_path=str(dataset_root / "sample_submission.csv"),
        train_hdf5_path=str(dataset_root / "train-image.hdf5"),
        test_hdf5_path=str(dataset_root / "test-image.hdf5"),
    )

## 3. Cấu hình huấn luyện Image-Only

In [ ]:
OUTPUT_DIR = "/kaggle/working/image_output" if is_kaggle else "./mid_fusion_lite_outputs/image_only"

image_config = MidFusionLiteConfig(
    preset_name="image_only_train",
    seed=42,

    # --- Cross-Validation ---
    n_splits=5,
    fold_indices=None,       # None = chạy hết 5 fold; [0] = chạy thử 1 fold

    # --- Chế độ Image-Only ---
    run_mode="image_only",
    use_tabm_tabular=False,         # Không cần nhánh tabular

    # --- Cấu hình ảnh ---
    backbone_name="efficientnet_b0",
    image_size=224,
    stage2_train_backbone=True,        # Fine-tune backbone
    download_pretrained_backbone=False,  # Tải trọng số ImageNet của timm (cần Internet)
    backbone_weights_path=None,       # Hoặc trỏ tới file .pth offline nếu Kaggle no-internet

    # --- Siêu tham số huấn luyện ---
    stage2_epochs=30,
    warmup_epochs=3,
    patience=4,
    lr=5e-4,
    min_lr=1e-6,
    weight_decay=1.32e-4,
    gradient_clipping_norm=1.0,
    label_smoothing=0.01,

    # --- Batch & sampler ---
    train_batch_size=64,
    valid_batch_size=64,
    negative_ratio=1,        # Undersample âm tính mỗi epoch cho cân bằng lớp
    num_workers=4 if is_kaggle else 0,

    # --- Tối ưu & xuất ---
    use_amp=True,
    save_checkpoints=True,
    output_dir=OUTPUT_DIR,
)

image_config = resolve_isic_data_paths(image_config)
print("[+] Cấu hình sẵn sàng. Output ->", image_config.output_dir)


[+] Đang dò bộ dữ liệu ISIC 2024 trong /kaggle/input ...
    -> Tìm thấy dữ liệu tại: /kaggle/input/competitions/isic-2024-challenge
[+] Cấu hình sẵn sàng. Output -> /kaggle/working/image_output


### (Tuỳ chọn) Chạy thử nhanh — smoke test

Bỏ comment cell dưới để chạy **1 fold, 2 epoch, giới hạn số step** nhằm kiểm tra pipeline trước khi train full.

In [5]:
# image_config = image_config.with_updates(
#     fold_indices=[0],
#     stage2_epochs=2,
#     warmup_epochs=0,
#     max_train_steps_per_epoch=60,
#     max_valid_steps=40,
# )
# print("[+] Đã bật chế độ smoke test (1 fold, 2 epoch).")

## 4. Chạy huấn luyện Cross-Validation

In [6]:
print("=" * 60)
print("  TRAIN IMAGE-ONLY — MID FUSION LITE (ISIC 2024)")
print("=" * 60)

results = run_mid_fusion_cv(image_config)

print("\n" + "=" * 60)
print("  KẾT QUẢ HUẤN LUYỆN")
print("=" * 60)
cv_pauc = results.get("cv_pauc", None)
print(f"  CV pAUC : {cv_pauc:.5f}" if cv_pauc is not None else "  CV pAUC : N/A")
print("\n  Chi tiết từng fold:")
for i, m in enumerate(results.get("fold_metrics", [])):
    print(f"   - Fold {i}: {m}")
print("=" * 60)

  TRAIN IMAGE-ONLY — MID FUSION LITE (ISIC 2024)
Running preset: image_only_train
Device: CUDA
Output directory: /kaggle/working/image_output
Numeric features: 609
Categorical features: 5
Fold column 'fold' not found. Generating fold splits dynamically using StratifiedGroupKFold.
Selected folds: [0, 1, 2, 3, 4]
Fold 1/5


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


  pos_weight_full=32.606 (Stage 1 no-sampler) | pos_weight_sampled=1.000 (Stage 2/3 with sampler)
Loaded local backbone weights: /kaggle/input/datasets/khiemdangle/backbone-weight/efficientnet_b0_backbone.pth | missing=0 | unexpected=0
Epoch 01 | train_loss=0.68248 | valid_loss=0.73389 | valid_auroc=0.64888 | valid_pauc=0.02851
Epoch 02 | train_loss=0.58433 | valid_loss=0.75147 | valid_auroc=0.84342 | valid_pauc=0.09352
Epoch 03 | train_loss=0.50065 | valid_loss=0.61637 | valid_auroc=0.86517 | valid_pauc=0.09674
Epoch 04 | train_loss=0.47924 | valid_loss=0.51167 | valid_auroc=0.88162 | valid_pauc=0.11322
Epoch 05 | train_loss=0.40902 | valid_loss=0.29497 | valid_auroc=0.88441 | valid_pauc=0.12528
Epoch 06 | train_loss=0.42845 | valid_loss=0.27609 | valid_auroc=0.90391 | valid_pauc=0.13148
Epoch 07 | train_loss=0.39688 | valid_loss=0.32100 | valid_auroc=0.90213 | valid_pauc=0.13117
Epoch 08 | train_loss=0.38106 | valid_loss=0.27188 | valid_auroc=0.90783 | valid_pauc=0.13645
Epoch 09 | t

## 5. Xuất submission & liệt kê checkpoint

In [7]:
# Kaggle yêu cầu file nộp tên đúng 'submission.csv' ở /kaggle/working/
sub_src = results.get("submission_path")
if sub_src and os.path.exists(sub_src):
    print(f"[+] Submission gốc: {sub_src}")
    if is_kaggle:
        dst = "/kaggle/working/submission.csv"
        shutil.copy(sub_src, dst)
        print(f"[+] Đã copy ra: {dst}")
else:
    print("[!] Không tìm thấy submission_path trong kết quả.")

# Liệt kê các checkpoint đã lưu
ckpt_dir = Path(image_config.output_dir)
if ckpt_dir.exists():
    ckpts = sorted(ckpt_dir.rglob("*.pt"))
    print(f"\n[+] {len(ckpts)} checkpoint trong {ckpt_dir}:")
    for p in ckpts:
        print("   -", p)

[+] Submission gốc: /kaggle/working/image_output/submission_mid_fusion_lite.csv
[+] Đã copy ra: /kaggle/working/submission.csv

[+] 5 checkpoint trong /kaggle/working/image_output:
   - /kaggle/working/image_output/mid_fusion_lite_fold_0.pt
   - /kaggle/working/image_output/mid_fusion_lite_fold_1.pt
   - /kaggle/working/image_output/mid_fusion_lite_fold_2.pt
   - /kaggle/working/image_output/mid_fusion_lite_fold_3.pt
   - /kaggle/working/image_output/mid_fusion_lite_fold_4.pt


In [8]:
# Xem trước OOF predictions (nếu có)
oof_df = results.get("oof_df")
if oof_df is not None:
    print(f"[+] OOF: {len(oof_df)} dòng — lưu tại {results.get('oof_path')}")
    try:
        display(oof_df.head())
    except NameError:
        print(oof_df.head())
else:
    print("Không có OOF trong results (key 'oof_df').")

[+] OOF: 401059 dòng — lưu tại /kaggle/working/image_output/oof_predictions.csv


,isic_id,target,oof_pred,used_in_cv
0,ISIC_0015670,0.0,0.000869,1
1,ISIC_0015845,0.0,0.410400,1
2,ISIC_0015864,0.0,0.002958,1
3,ISIC_0015902,0.0,0.040466,1
4,ISIC_0024200,0.0,0.018188,1
